# FieldShift Mini — browser language-model training and evaluation

This is the actual compact model used by the current app: a **word-level autoregressive recurrent language model**, trained with next-word cross-entropy. It is not a topic-classifier output layer or a general-purpose large language model. Production only uses its guarded response-topic selection with traceable retrieval. Raw generated words never become scientific advice.

The model has 39,274 parameters and 157,096 bytes of float32 weights. Training used browser TensorFlow.js 4.22.0 / WASM, one thread, 100 epochs, seed-controlled initializers and no shuffling. Checkpoint selection used validation matches, then validation loss. Run evidence is inspected below.

Node starts an isolated local test page and browser automation; training and production inference happen in the browser. No Python or Node inference server is deployed on Vercel.


In [ ]:
from pathlib import Path
import json, hashlib, subprocess, sys
ROOT = Path.cwd() if (Path.cwd() / "server").exists() else Path.cwd().parent
DIRECTORY = ROOT / "public/models/fieldshift-mini"
corpus = json.loads((DIRECTORY / "corpus.json").read_text(encoding="utf-8"))
evaluation = json.loads((DIRECTORY / "evaluation.json").read_text(encoding="utf-8"))
tokenizer = json.loads((DIRECTORY / "tokenizer.json").read_text(encoding="utf-8"))
topology = json.loads((DIRECTORY / "model.json").read_text(encoding="utf-8"))
weights = (DIRECTORY / "weights.bin").read_bytes()
assert len(weights) == evaluation["weightBytes"]
print({"python": sys.version, "parameters": evaluation["parameters"], "weight_bytes": len(weights), "sha256": hashlib.sha256(weights).hexdigest()})


## Data provenance and split limits

The corpus contains development-only **synthetic question wording**, not agronomist-reviewed farmer data or fabricated NASA observations. Vocabulary uses training rows only; template splits still share topic vocabulary. Accepted results: 85/126 overall and 14/36 additional fixed free-form questions, versus 83/126 and 8/36 before. Legacy matches decreased 75/90 to 71/90. Two candidates were evaluated; these are development comparisons, not an independent untouched study or agronomic accuracy.

The full token vocabulary is built from the development corpus, including held-out wording; this is another limitation. A production experiment needs a vocabulary learned only from training data, truly independent farmer questions, region/source grouping, out-of-scope tests and independent review. The app does not treat this experiment as reliable free-form advice.


In [ ]:
rows = corpus["rows"]
print(corpus["kind"])
print(corpus["limitation"])
print({name: sum(row["split"] == name for row in rows) for name in ["train", "validation", "test"]})
print({"test_correct": evaluation["correct"], "test_total": evaluation["testExamples"], "freeform_correct": evaluation["freeformCorrect"], "freeform_total": evaluation["freeformExamples"]})
print("Free-form failures:")
for row in evaluation["results"]:
    if row["freeform"] and row["expected"] != row["predicted"]:
        print(row)
print("Corpus SHA-256:", hashlib.sha256((DIRECTORY / "corpus.json").read_bytes()).hexdigest())


## Architecture and objective

Vocabulary embedding (24 dimensions) → LSTM (48 units, sequence output) → Dense softmax over **words**, not topic labels. The model predicts the next word at each timestep; padding is masked in the loss. It can autoregressively generate a continuation, but raw generation is only exercised in developer evaluation.

At runtime, nine source-bound explanatory continuations are scored by answer-word negative log likelihood. An explicit lexical and score-gap guard permits a topic selection only when supported; otherwise retrieval is left unchanged. The evidence answer renders before neural inference completes. This restriction prevents generated NASA readings, crop claims and citations from entering the app.

Artifacts are ordinary TensorFlow.js model topology, weight specs and binary weights. This exact training → saved artifact → browser inference combination was executed. It is separate from the optional, unexecuted Keras/TF.js intent-classifier and pretrained fine-tuning notebooks.


In [ ]:
print("Vocabulary:", len(tokenizer["vocabulary"]))
for layer in topology["modelTopology"]["config"]["layers"]:
    config = layer["config"]
    print(layer["class_name"], {k: config[k] for k in ["input_dim", "output_dim", "units", "activation", "recurrent_activation", "return_sequences"] if k in config})
specs = topology["weightsManifest"][0]["weights"]
print("Weight tensors:", [(s["name"], s["shape"], s["dtype"]) for s in specs])


## Reproduce the real browser training run

Requires Node.js 22+, installed Chrome/Chromium/Edge and internet for pinned TensorFlow.js/WASM runtime files. The scripts never download a browser or invoke a cloud model. Training uses the small development corpus, not a pretrained weight download. It takes several minutes on the observed development machine; no hardware-independent guarantee is made.

TF.js WASM lacked UnsortedSegmentSum for embedding gradients; the development-only kernel was added and tested. Retraining writes artifacts/mini-candidate only. The quality gate compares against the current shipped model; an equal reproduction can exit 1 because it is not a new improvement. Review results before separately promoting files and updating integrity hashes. Training is off by default.


In [ ]:
RETRAIN = False
if RETRAIN:
    subprocess.run(["node", "scripts/build-mini-corpus.js"], cwd=ROOT, check=True)
    subprocess.run(["node", "scripts/train-mini-lm.js"], cwd=ROOT, check=True)
    print("Inspect artifacts/mini-candidate; shipped weights were not replaced.")
else:
    print("Retraining disabled. Existing trained artifacts and recorded evaluations were inspected only.")


## TensorFlow/Keras reproduction path

An equivalent task can be explored in Python TensorFlow/Keras using Embedding(24), LSTM(48, return_sequences=True), Dense(vocabulary_size, softmax), next-word targets and masked padding loss. Pin the Python/TensorFlow/Keras versions; match token IDs and recurrent activations explicitly. TF.js and Keras defaults can differ, so do not assume saved-weight equivalence or claim a converter passed. The demonstrated shipped path is browser TensorFlow.js training and browser inference.

Use si_experiments.ipynb for a separate Keras intent-classification laboratory, and pretrained_llm_finetuning.ipynb for the guarded DistilGPT2 study. Neither is the shipped compact model or a completed pretrained fine-tune.


## Actual latency, memory and safety evidence

The local benchmark report records first evidence response latency, cold/warm model initialization, neural inference, tensor bytes and WASM heap capacity. It does **not** establish full peak browser/process memory or real mobile performance. Inspect a new run before reporting new figures. Source links and limitations remain attached regardless of model selection.


In [ ]:
report = ROOT / "artifacts/mini-lm-benchmark.json"
if report.exists():
    benchmark = json.loads(report.read_text(encoding="utf-8"))
    print(json.dumps(benchmark, indent=2))
else:
    print("Run node scripts/benchmark-mini-lm.js after npm run build to produce a fresh benchmark.")
